# Braket Direct — n-target QPUF analysis

Reads the jobs in `job_results/job_log.txt` and the retrieved counts in
`job_results/<uuid>.json` (produced by `checkRetrieve.py`), then analyses
the two-stage QPE QPUF results for `n_targ >= 1`.

Modeled on `garnet-approxQPEQPUF-1target/result_analysis.ipynb`, generalised
to handle multi-target Haar-random unitaries: each `U` has `2^n_targ`
eigenphases, so the ideal QPE peaks form a set rather than a single bin.

In [1]:
import json
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

JOB_RESULTS_DIR = os.path.join(os.getcwd(), 'job_results')
LOG_FILE        = os.path.join(JOB_RESULTS_DIR, 'job_log.txt')

print('job_results dir :', JOB_RESULTS_DIR)
print('job_log.txt     :', LOG_FILE, ' (exists:', os.path.exists(LOG_FILE), ')')

job_results dir : /Users/user/python/QUEN_QPUF/braket_direct_reserve/job_results
job_log.txt     : /Users/user/python/QUEN_QPUF/braket_direct_reserve/job_results/job_log.txt  (exists: True )


## Analysis helpers

In [2]:
def decode_unitary(rec: dict) -> np.ndarray:
    """Reconstruct the complex Haar U from the serialized real/imag lists."""
    u = rec['unitary']
    return np.array(u['real']) + 1j * np.array(u['imag'])


def parse_outcome(bitstring: str, n_prec: int) -> tuple[int, int]:
    """Counts key -> (m1, m2). Handles 'c2 c1' or concatenated layout."""
    parts = bitstring.split(' ')
    if len(parts) == 2:
        return int(parts[1], 2), int(parts[0], 2)
    return int(bitstring[n_prec:], 2), int(bitstring[:n_prec], 2)


def cyclic_distance(a: int, b: int, n_prec: int) -> int:
    M    = 2 ** n_prec
    diff = abs(a - b)
    return min(diff, M - diff)


def eigenphase_bins(U: np.ndarray, n_prec: int) -> list[int]:
    """Sorted list of distinct ideal QPE bins from eig(U)."""
    eigvals = np.linalg.eig(U)[0]
    phases  = np.mod(np.angle(eigvals) / (2 * np.pi), 1.0)
    return sorted({round(float(p) * 2 ** n_prec) % 2 ** n_prec for p in phases})


def nearest_eigenbin_distance(m: int, bins: list[int], n_prec: int) -> int:
    """Cyclic distance from m to the nearest ideal eigenbin."""
    return min(cyclic_distance(m, b, n_prec) for b in bins)


def analyse_counts(counts: dict, n_prec: int, U: np.ndarray, delta: int = 2,
                   label: str = '') -> dict:
    """Print acceptance stats and top outcomes. Returns a stats dict."""
    bins  = eigenphase_bins(U, n_prec)
    total = sum(counts.values())

    consistent = 0   # m1 ≈ m2
    near_eig   = 0   # both m1 and m2 near some eigenbin
    for outcome, cnt in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        if cyclic_distance(m1, m2, n_prec) <= delta:
            consistent += cnt
        if (nearest_eigenbin_distance(m1, bins, n_prec) <= delta and
            nearest_eigenbin_distance(m2, bins, n_prec) <= delta):
            near_eig += cnt

    prefix = f'[{label}] ' if label else ''
    print(f'{prefix}Total shots                : {total}')
    print(f'{prefix}|m1 - m2|_cyclic <= {delta}      : {consistent} ({consistent/total:.4f})')
    print(f'{prefix}both near eigenbin (<= {delta}) : {near_eig} ({near_eig/total:.4f})')
    print(f'{prefix}Ideal eigenbins             : {bins}')

    print(f'\n{prefix}Top 10 outcomes:')
    print(f'  {"bitstring":28s}  m1   m2   dist  count')
    print(f'  {"-"*54}')
    for k, v in sorted(counts.items(), key=lambda x: -x[1])[:10]:
        m1, m2 = parse_outcome(k, n_prec)
        print(f'  {k!r:28s}  {m1:4d} {m2:4d}  {cyclic_distance(m1, m2, n_prec):4d}  {v}')

    return {
        'total':       total,
        'consistent':  consistent,
        'near_eig':    near_eig,
        'bins':        bins,
        'delta':       delta,
    }

print('Analysis helpers loaded.')

Analysis helpers loaded.


## Plotting helpers

In [3]:
def plot_m1_vs_m2(counts: dict, n_prec: int, U: np.ndarray, n_shots: int,
                   title_suffix: str = '', ax=None):
    """Scatter of (m1, m2) with dashed gridlines at every ideal eigenbin."""
    m1_vals, m2_vals, weights = [], [], []
    for outcome, cnt in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        m1_vals.append(m1); m2_vals.append(m2); weights.append(cnt)

    bins = eigenphase_bins(U, n_prec)
    M    = 2 ** n_prec

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    s = np.array(weights) * (200.0 / max(weights))
    ax.scatter(m1_vals, m2_vals, s=s, alpha=0.6)
    ax.plot([0, M - 1], [0, M - 1], 'r--', alpha=0.5, label='m1 = m2')
    for b in bins:
        ax.axvline(b, color='green', linestyle=':', alpha=0.4)
        ax.axhline(b, color='green', linestyle=':', alpha=0.4)
    ax.set_xlim(-0.5, M - 0.5); ax.set_ylim(-0.5, M - 0.5)
    ax.set_xlabel('m1  (Stage 1 QPE)')
    ax.set_ylabel('m2  (Stage 2 QPE)')
    title = f'm1 vs m2 (n_prec={n_prec}, shots={n_shots})'
    if title_suffix:
        title += f'  -  {title_suffix}'
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=8)
    return ax


def plot_acceptance_vs_delta(counts: dict, n_prec: int, n_shots: int,
                              delta_mark: int = 2, ax=None,
                              title_suffix: str = ''):
    """Acceptance rate (|m1-m2|_cyclic <= delta) vs delta."""
    total       = sum(counts.values())
    delta_range = list(range(0, 2 ** n_prec + 1))
    acc_rates   = [
        sum(cnt for outcome, cnt in counts.items()
            if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= d) / total
        for d in delta_range
    ]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_range, acc_rates, marker='o', markersize=3)
    ax.axvline(delta_mark, color='r', linestyle='--', label=f'delta={delta_mark}')
    ax.set_xlabel('delta  (cyclic distance threshold)')
    ax.set_ylabel('acceptance rate')
    title = f'Acceptance vs delta (n_prec={n_prec}, shots={n_shots})'
    if title_suffix:
        title += f'  -  {title_suffix}'
    ax.set_title(title)
    ax.legend()
    return ax


def plot_hw_vs_sim(hw: dict, sim: dict, U: np.ndarray):
    """2x2 comparison: scatter and delta-curve, hardware vs local simulator."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 11))
    plot_m1_vs_m2(hw['counts'],  hw['n_prec'],  U, hw['n_shots'],
                   title_suffix=hw.get('label', 'Hardware'),       ax=axes[0, 0])
    plot_m1_vs_m2(sim['counts'], sim['n_prec'], U, sim['n_shots'],
                   title_suffix=sim.get('label', 'Simulator'),      ax=axes[0, 1])
    plot_acceptance_vs_delta(hw['counts'],  hw['n_prec'],  hw['n_shots'],
                              delta_mark=hw.get('delta', 2),
                              title_suffix=hw.get('label', 'Hardware'),  ax=axes[1, 0])
    plot_acceptance_vs_delta(sim['counts'], sim['n_prec'], sim['n_shots'],
                              delta_mark=sim.get('delta', 2),
                              title_suffix=sim.get('label', 'Simulator'), ax=axes[1, 1])
    plt.tight_layout()
    return fig

print('Plot helpers loaded.')

Plot helpers loaded.


## Local-simulator helper (re-runs the same `U` on AerSimulator)

Reuses the exact circuit builders from `submit_QPUF_ntarg.py` so the
hardware and simulator runs are byte-identical at the circuit level.

In [4]:
from submit_QPUF_ntarg import build_full_circuit
from qiskit import transpile

def run_local_sim(rec: dict, U: np.ndarray, n_shots: int | None = None,
                  delta: int = 2) -> dict:
    """Run the same circuit (same U, same target-init seed) on AerSimulator."""
    try:
        from qiskit_aer import AerSimulator
    except ImportError as e:
        raise RuntimeError('qiskit-aer not installed') from e

    n_prec   = rec['n_prec']
    n_targ   = rec['n_targ']
    init_sd  = rec.get('target_init_seed', 99)
    shots    = n_shots if n_shots is not None else rec['n_shots']

    qc = build_full_circuit(n_prec, n_targ, U, target_init_seed=init_sd)
    sim = AerSimulator()
    qc_sim = transpile(qc, sim)
    counts = sim.run(qc_sim, shots=shots).result().get_counts()

    return {
        'counts':  counts,
        'n_prec':  n_prec,
        'n_targ':  n_targ,
        'n_shots': shots,
        'delta':   delta,
        'label':   'Local Simulator',
    }

print('Local-sim helper loaded.')

Local-sim helper loaded.


## List submitted jobs (from `job_log.txt`)

In [5]:
with open(LOG_FILE) as f:
    submitted = [json.loads(line) for line in f if line.strip()]

print(f'{len(submitted)} job(s) recorded in job_log.txt:\n')
print(f"  {'idx':>3}  {'datetime':25s}  {'qpu':20s}  {'type':18s}  n_prec  n_targ  n_shots  uuid")
print('  ' + '-' * 110)
for i, r in enumerate(submitted):
    uuid = r['job_id'].split('/')[-1]
    print(f"  {i:>3}  {r['datetime'][:25]:25s}  {r['qpu']:20s}  "
          f"{r.get('circuit_type','-'):18s}  {r['n_prec']:>6}  {r.get('n_targ','-'):>6}  "
          f"{r['n_shots']:>7}  {uuid[:8]}...")

0 job(s) recorded in job_log.txt:

  idx  datetime                   qpu                   type                n_prec  n_targ  n_shots  uuid
  --------------------------------------------------------------------------------------------------------------


## List retrieved results (`job_results/*.json` written by `checkRetrieve.py`)

In [6]:
retrieved_paths = sorted(glob.glob(os.path.join(JOB_RESULTS_DIR, '*.json')))
retrieved = []
for p in retrieved_paths:
    with open(p) as f:
        retrieved.append(json.load(f))

print(f'{len(retrieved)} retrieved result(s):\n')
for i, r in enumerate(retrieved):
    uuid = r['job_id'].split('/')[-1]
    print(f"  [{i}]  {uuid[:8]}...  {r.get('circuit_type','-')}  "
          f"n_prec={r['n_prec']}  n_targ={r.get('n_targ','-')}  "
          f"shots={r['n_shots']}  unique_outcomes={len(r['counts'])}")

if not retrieved:
    print('(none yet — run `python checkRetrieve.py` after the job completes)')

0 retrieved result(s):

(none yet — run `python checkRetrieve.py` after the job completes)


## Pick one retrieved job and analyse

In [7]:
# Choose which retrieved job to analyse. Default: most recent.
JOB_IDX = -1
DELTA   = 2

assert retrieved, 'No retrieved results yet. Run checkRetrieve.py first.'
rec     = retrieved[JOB_IDX]
U       = decode_unitary(rec)
counts  = rec['counts']
n_prec  = rec['n_prec']
n_targ  = rec['n_targ']
n_shots = rec['n_shots']
uuid    = rec['job_id'].split('/')[-1]

print(f'Job UUID    : {uuid}')
print(f'Submitted   : {rec["datetime"]}')
print(f'QPU         : {rec["qpu"]}')
print(f'Type        : {rec.get("circuit_type","-")}')
print(f'n_prec      : {n_prec}')
print(f'n_targ      : {n_targ}  (U is {2**n_targ}×{2**n_targ})')
print(f'shots       : {n_shots}')
print(f'transp gates: {rec.get("n_gates","-")}')
print(f'per_U_gates : {rec.get("per_U_gates","-")}  (transpiled gates per ctrl-U)')
u_err = float(np.max(np.abs(U.conj().T @ U - np.eye(U.shape[0]))))
print(f'|U†U − I|   : {u_err:.2e}  (sanity: stored U is unitary)')

AssertionError: No retrieved results yet. Run checkRetrieve.py first.

In [ ]:
stats_hw = analyse_counts(counts, n_prec, U, delta=DELTA, label='hw')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_m1_vs_m2(counts, n_prec, U, n_shots, title_suffix='Hardware', ax=axes[0])
plot_acceptance_vs_delta(counts, n_prec, n_shots, delta_mark=DELTA,
                          title_suffix='Hardware', ax=axes[1])
plt.tight_layout(); plt.show()

## Hardware vs Local Simulator

Rebuilds the same circuit with the stored `U` and runs it on AerSimulator,
so any divergence between the two panels is attributable to device noise
(not to a circuit-construction bug).

In [ ]:
sim = run_local_sim(rec, U, n_shots=max(n_shots, 1000), delta=DELTA)
stats_sim = analyse_counts(sim['counts'], n_prec, U, delta=DELTA, label='sim')

hw = {
    'counts':  counts, 'n_prec': n_prec, 'n_targ': n_targ,
    'n_shots': n_shots, 'delta': DELTA, 'label': f'Hardware ({rec["qpu"]})',
}
fig = plot_hw_vs_sim(hw, sim, U); plt.show()